# 03 — Modeling v1 results

Seed-only LR baseline, XGBoost v1, and Platt calibration on leave-one-tournament-out (16 folds). Outputs live under `data/outputs/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

ROOT = Path("..").resolve()
PRED = ROOT / "data" / "outputs" / "predictions"
IMP = ROOT / "data" / "training" / "feature_importance_log"
CAL_PLOT = ROOT / "data" / "outputs" / "calibration_plot.png"
MATCHUP = ROOT / "data" / "features" / "matchup_features.parquet"

bl = pd.read_parquet(PRED / "baseline_lr_predictions.parquet")
xgb = pd.read_parquet(PRED / "xgb_v1_predictions.parquet")
cal = pd.read_parquet(PRED / "xgb_v1_calibrated_predictions.parquet")
matchup = pd.read_parquet(MATCHUP, columns=["year", "round", "t1_team_norm", "t2_team_norm", "t1_seed", "t2_seed"])

## Section 1 — Model comparison (mean ± std, 16 LOO folds)

Metrics computed on held-out tournament rows per `test_year`.

In [ ]:
def fold_table(df: pd.DataFrame, prob_col: str) -> pd.DataFrame:
    rows = []
    for y, g in df.groupby("test_year"):
        yt = g["result"].to_numpy(dtype=int)
        p = g[prob_col].to_numpy(dtype=float)
        pr = (p >= 0.5).astype(int)
        auc = np.nan
        if len(np.unique(yt)) > 1:
            auc = roc_auc_score(yt, p)
        rows.append(
            {
                "test_year": int(y),
                "accuracy": accuracy_score(yt, pr),
                "log_loss": log_loss(yt, p, labels=[0, 1]),
                "brier": brier_score_loss(yt, p),
                "auc_roc": auc,
            }
        )
    return pd.DataFrame(rows)


def summarize(ft: pd.DataFrame) -> pd.Series:
    m = ft.mean(numeric_only=True)
    s = ft.std(numeric_only=True)
    return pd.Series(
        {
            "accuracy": f"{m['accuracy']:.3f} ± {s['accuracy']:.3f}",
            "log_loss": f"{m['log_loss']:.3f} ± {s['log_loss']:.3f}",
            "brier": f"{m['brier']:.3f} ± {s['brier']:.3f}",
            "auc_roc": f"{np.nanmean(ft['auc_roc']):.3f} ± {np.nanstd(ft['auc_roc']):.3f}",
        }
    )


ft_lr = fold_table(bl, "predicted_prob_t1")
ft_xgb = fold_table(xgb, "xgb_raw_prob_t1")
ft_cal = fold_table(cal, "calibrated_prob_t1")

cmp = pd.DataFrame(
    {
        "Seed-only LR": summarize(ft_lr),
        "XGBoost v1": summarize(ft_xgb),
        "XGBoost + Platt": summarize(ft_cal),
    }
).T
cmp.columns = ["Accuracy", "Log-loss", "Brier", "AUC-ROC"]
display(cmp)

## Section 2 — Feature importance (top 30 by mean gain)

Colors follow **name heuristics** mapped to the requested groups (schema `feature_group` is mostly `matchup`/`metadata` for these columns).

In [ ]:
def viz_bucket(name: str) -> tuple[str, str]:
    n = name.lower()
    if "coach" in n:
        return "coach", "#ff7f0e"
    if "seed_prior" in n or "historical_win" in n or n.startswith("historical_"):
        return "historical_prior", "#d62728"
    if any(
        k in n
        for k in (
            "streak",
            "days_since",
            "early_season",
            "late",
            "win_rate",
            "momentum",
        )
    ):
        return "momentum_rolling", "#2ca02c"
    if "depth" in n or "player" in n:
        return "player_quality", "#9467bd"
    if n == "round" or "bubble" in n or "flag" in n or "tempo" in n or "midmajor" in n or "major_conf" in n:
        return "metadata", "#7f7f7f"
    return "efficiency_matchup", "#1f77b4"


frames = []
for p in sorted(IMP.glob("*_importance.csv")):
    d = pd.read_csv(p)
    d["fold"] = p.stem.split("_")[0]
    frames.append(d)
imp_long = pd.concat(frames, ignore_index=True)
mean_gain = imp_long.groupby("feature", as_index=False)["gain"].mean().sort_values("gain", ascending=False)
top30 = mean_gain.head(30).copy()
top30["bucket"], top30["color"] = zip(*top30["feature"].map(viz_bucket))

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top30))
ax.barh(y_pos, top30["gain"], color=top30["color"])
ax.set_yticks(y_pos)
ax.set_yticklabels(top30["feature"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Mean gain (XGBoost, 16 folds)")
ax.set_title("Top 30 features by mean gain")
from matplotlib.patches import Patch

legend_el = [
    Patch(facecolor="#1f77b4", label="efficiency_matchup"),
    Patch(facecolor="#2ca02c", label="momentum_rolling"),
    Patch(facecolor="#ff7f0e", label="coach"),
    Patch(facecolor="#9467bd", label="player_quality"),
    Patch(facecolor="#d62728", label="historical_prior"),
    Patch(facecolor="#7f7f7f", label="metadata"),
]
ax.legend(handles=legend_el, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Section 3 — Per-year accuracy (XGB v1)

Annotations: approximate ratings (2008–2010), 2021 bubble, notable Cinderella runs.

In [ ]:
acc_by_year = (
    xgb.assign(hit=(xgb["predicted_winner"] == xgb["result"]).astype(int))
    .groupby("test_year")["hit"]
    .mean()
    .sort_index()
)
mean_acc = float(acc_by_year.mean())

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(acc_by_year.index.astype(int), acc_by_year.values, color="steelblue")
ax.axhline(mean_acc, color="crimson", linestyle="--", label=f"Mean = {mean_acc:.2f}")
ax.set_xlabel("Held-out tournament year")
ax.set_ylabel("Accuracy")
ax.set_title("LOO test accuracy by year (XGBoost v1)")
ax.legend()
for x, txt in [
    (2021, "bubble"),
    (2009, "2008-10\napprox ratings"),
    (2023, "FAU F4"),
    (2022, "St. Peter's E8"),
    (2018, "UMBC"),
]:
    if x in acc_by_year.index:
        ax.annotate(txt, xy=(x, acc_by_year.loc[x]), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=7, rotation=0)
plt.tight_layout()
plt.show()

## Section 4 — Round-level accuracy (XGB v1, pooled LOO tests)

Hypothesis: accuracy rises in later rounds as weaker teams exit.

In [ ]:
ROUND_LABEL = {
    0: "First Four",
    1: "Round of 64",
    2: "Round of 32",
    3: "Sweet 16",
    4: "Elite Eight",
    5: "Final Four",
    6: "Championship",
}
xr = xgb.assign(hit=(xgb["predicted_winner"] == xgb["result"]).astype(int))
by_r = xr.groupby("round")["hit"].mean().sort_index()
labels = [ROUND_LABEL.get(int(i), str(i)) for i in by_r.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, by_r.values, color="seagreen")
ax.set_ylabel("Accuracy")
ax.set_title("Pooled LOO accuracy by tournament round")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
display(by_r.to_frame("accuracy"))

## Section 5 — Calibration reliability diagram

**Good calibration:** points hug the diagonal — predicted probabilities match observed frequencies. **Poor calibration:** systematic curve above/below the line indicates under- or over-confidence.

In [ ]:
display(Image(filename=str(CAL_PLOT)))

## Section 6 — Upset detection (calibrated probs)

Games where the **seed underdog** had **below 30%** model win probability (from the underdog’s perspective).

In [ ]:
cal_m = cal.merge(
    matchup,
    on=["year", "round", "t1_team_norm", "t2_team_norm"],
    how="left",
)


def underdog_win_prob(row) -> float:
    s1, s2 = row["t1_seed"], row["t2_seed"]
    p_t1 = row["calibrated_prob_t1"]
    if pd.isna(s1) or pd.isna(s2):
        return float("nan")
    if s1 < s2:
        return float(1.0 - p_t1)
    if s2 < s1:
        return float(p_t1)
    return float("nan")


def underdog_won(row) -> bool:
    s1, s2 = row["t1_seed"], row["t2_seed"]
    r = int(row["result"])
    if pd.isna(s1) or pd.isna(s2) or s1 == s2:
        return False
    if s1 < s2:
        return r == 0
    return r == 1


cal_m["p_underdog"] = cal_m.apply(underdog_win_prob, axis=1)
cal_m["underdog_won"] = cal_m.apply(underdog_won, axis=1)
mask = cal_m["p_underdog"] < 0.3
sub = cal_m.loc[mask].copy()
print(f"Games with P(underdog win) < 30%: {len(sub)}")
print(f"Underdog actually won: {int(sub['underdog_won'].sum())}")
display(sub.groupby("round").agg(n=("result", "size"), underdog_wins=("underdog_won", "sum")))
sub["seed_pair"] = sub.apply(
    lambda r: tuple(sorted([int(r["t1_seed"]), int(r["t2_seed"])])), axis=1
)
display(sub.groupby("seed_pair").agg(n=("result", "size"), ud_wins=("underdog_won", "sum")).head(15))

miss = sub[sub["underdog_won"]].copy()
miss = miss.sort_values("p_underdog")
print("Biggest misses (underdog won, model most confident favorite):")
display(
    miss[
        [
            "test_year",
            "round",
            "t1_team_norm",
            "t2_team_norm",
            "t1_seed",
            "t2_seed",
            "calibrated_prob_t1",
            "p_underdog",
        ]
    ].head(15)
)

## Section 7 — Interpretation

**Versus seed baseline:** XGBoost v1 improves mean LOO accuracy by roughly **20 percentage points** and AUC by **~0.19** versus seed-only logistic regression — well above a 0.02 AUC rule-of-thumb. **Sanity check:** these headline numbers are strong enough that an independent **leakage audit** (feature timing, duplicate rows, label alignment) should be on the v2 checklist.

**Feature groups:** Top gains cluster on efficiency differentials (`delta_*` ratings-style columns), with coach experience, seed priors, and rolling/momentum deltas appearing in the long tail of the top-30 bar chart.

**Calibration:** Platt scaling on raw XGB probabilities **slightly increased** mean Brier and log-loss in LOO evaluation here (raw scores were already fairly sharp). For bracket simulation, re-check calibration on a holdout or use isotonic regression in v2 if reliability remains off-diagonal.

**v2 priorities:** (1) Audit for subtle leakage if metrics stay unrealistically high vs published baselines. (2) Try temperature scaling / isotonic calibration. (3) Add interaction terms or monotonic constraints for seeds. (4) Stress-test bubble / Cinderella years called out above.

**Limitations:** Small *n* per fold (~64–67 games); LOO variance is visible year-to-year. Platt is fit per fold on in-fold training predictions — test calibration can still drift. Heuristic feature colors approximate groups because many registry entries share the generic `matchup` label.